# Business Entity Resolution — EDA Summary

Training data: S1 = 2,206,821 records (deduplicated reference), S2 and S3 ≈ 5.3M each.
All figures from a strict read (`dtype=str, na_filter=False, keep_default_na=False,
quoting=QUOTE_NONE`). Ground-truth-conditioned figures sampled at n=20,000 pairs.

---

# FINDINGS

## 1.1 File structure

- Headers correct in all three sources; raw line count reconciles with parsed row count; no line deviates from 3 tabs. No embedded tabs or newlines shifting columns.
- `entity_id` unique and well-formed (`^S[123]-\d+$`) in every file, prefix always consistent with the file.
- **S1 confirmed deduplicated** — zero content duplicates. The problem statement's claim holds.

## 1.2 Missingness

| Source | Empty addresses | Literal `NA` names |
|---|---|---|
| S2 | 168,967 (~3%) | 2 |
| S3 | 175,916 (~3%) | 13 |

- No literal `"NaN"` / `"null"` / `"None"` text anywhere. The "thousands of NaN addresses" seen initially were **empty strings coerced by a default `read_csv`** — a reader artefact, not a data property.
- `punct_only = 0` in all sources. Addresses have essentially no short values (one `DL` in S3).
- Records unmatchable on both fields: **0 / 3 / 4** across S1 / S2 / S3. Nothing needs excluding.

## 1.3 Injected duplicates in S2 and S3

Identical `(name, address, country)` with distinct `entity_id`:

| Source | Duplicate rows | Groups | Sizes |
|---|---|---|---|
| S2 | 50,933 | 25,060 | 24,270×2, 769×3, 19×4, 2×5 |
| S3 | 37,241 | 18,381 | predominantly pairs |

Ground-truth cross-check returns `owners=1, unmatched=0` for **every group in both sources** — all members claimed by the same S1 entity, none unlabelled. Zero singletons among ~88k duplicate rows indicates deliberate injection, not coincidental collision.

## 1.4 Short names are initialisms — and they match

- ≤2-char names: 704 in S2, 9,275 in S3, with **98% / 99.9% having a true S1 match**. Excluding them would have discarded ~10k matchable entities.
- 3-letter names (`Boa`, `Zio`, `Ace`) are legitimate and appear in S1 too → degeneracy threshold is **≤2 chars, not ≤3**.
- Patterns: token initials skipping suffixes/connectors/numerics (`AC` ← Aditya Consulting Private Limited, `SO` ← Sudduth **and** Olea, `WD` ← **864** Washington Drive Company); first+last letter of single-token names, Title-cased (`Zx` ← Zaix); junk prefixes (`#K`, `@c`); diacritics (`MÍ` ← Morales Innovative).
- Address fallback viable — only 0.04% of short-name S3 records lack an address.

## 1.5 Address component order is not fixed

The apparent "38% missing house number" was **permuted addresses**, not missing numbers — the regex only inspected position 0.

```
OH, Columbus, 5559 Orville Avenue          state, city, street
Charlotte, NC, 833 Reliance Street         city, state, street
West Bengal, Howrah, 229, Kolkata, …       state, district, number, city, street
```

India is 78% of the gap against a 40% corpus baseline (~2.3× over-represented) because Indian addresses genuinely lead with door/building identifiers — but the US 22% is pure reordering. Widening the house-number regex gained only 62.2% → 65.4%; **order, not the regex, was the limit**.

## 1.6 Script distribution

| Source | latin | devanagari | gujarati | other |
|---|---|---|---|---|
| S1 | **1.0000** | — | — | — |
| S2 | 0.9058 | 0.0535 | 0.0061 | 0.0346 |
| S3 | 0.9473 | 0.0299 | 0.0034 | 0.0194 |

**S1 is 100% Latin** → transliteration is strictly one-way with a fixed target. `other` (3.5% / 1.9%) is accented Latin plus possible additional scripts.

## 1.7 Name token vocabulary (empirical, from S1)

- **Legal suffixes:** `limited` (521,915), `llc` (355,736), `inc` (238,306), `ltd`, `llp`, `corp`, `pc`, `pllc`, `lp`, `co`, `corporation`, `company`.
- **Business-type words, NOT legal forms:** `group`, `center`, `associates`, `partners`, `clinic`, `care`, `trust`, `holdings`, `society`, `school`, `foundation`, `church`, `academy`, `institute`, `services`, `medicine`. Stripping these collapses "Shree Medical Center" → "Shree".
- Bare `c` (45,559) and `d` (5,017) are tokenization artefacts from `P.C.` and `D.B.A.`.
- Leading tokens (`pediatric`, `blue`, `shree`, `golden`) are ordinary name words — no action.

## 1.8 Locality vocabulary (self-building, all positions)

US state codes (`tx`, `ny`, `nc`, `oh`) and Indian states/cities (`maharashtra`, `delhi`, `mumbai`, `bangalore`) emerge together from all-position segment counts. Notes: `mumbai` and `mumbai city` appear separately; `in` (49,472) is ambiguous between Indiana and India.

## 1.9 Landmark markers

Present in **4.6%** of S1 addresses — lower than the Indian sample suggested. Forms include `Opp.Rta Office`, `Near E.N.T Hospital` (no space after punctuation).

## 1.10 Match structure — the decisive numbers

```
0 matches : 123,247      6 : 164,868
1         : 119,157      7 :  63,968
2         : 375,212      8 :  18,680
3         : 530,841      9 :   4,205
4         : 484,115     10 :     534
5         : 321,957     11 :      37
```

- **Singleton rate: 5.58%** — far below what the problem statement's emphasis implied.
- Mean matches: **3.461**. p95 = 6, max = 11.
- Score from predicting all-empty: **0.056**. The conservative floor is worthless.
- Source split: 80.5% of entities match **both** S2 and S3 (1,776,047); S3-only 164,498; S2-only 143,029.

## 1.11 Separability — the problem is easier than it looks

| Feature | True p5 | True p50 | Random p50 | Random p95 |
|---|---|---|---|---|
| `name_ratio` | 11 | 88 | 32 | 47 |
| `name_token_set` | 12 | 100 | 32 | 49 |
| `addr_ratio` | **59** | 94 | 34 | **45** |

`addr_ratio` alone nearly separates the classes — true p5 (59) sits **above** random p95 (45).

## 1.12 The hard 12.5% is one identifiable pattern

True pairs with `name_token_set < 60`: 2,502 / 20,000 = **12.5%**. Of those:

- short b-name: **1.0%** → the initialism slice is *not* the difficulty
- address missing: **0.3%** → nor is missingness
- **numeric address overlap: 83.3%** → addresses agree on numbers even when names diverge

The hard cases are genuine name divergence (DBA/trade names, transliteration, heavy abbreviation) with a matching address.

## 1.13 Blocking recall ceilings

| Key | Recall |
|---|---|
| `name_token` | 0.8472 |
| `name_trigram` | 0.9110 |
| `addr_numeric` | 0.7990 |
| `addr_token` | **0.9576** |
| `name_token ∪ addr_numeric` | 0.9741 |
| `name_trigram ∪ addr_numeric` | **0.9854** |
| all four | **1.0000** |

Unreachable by any key: **0.01%**.

## 1.14 Two free structural constraints

- **Cross-country true pairs: 0 / 7,638,365.** Country is a hard partition.
- **S2/S3 IDs claimed by >1 S1 entity: 0.** Matching is a clean partition — every S2/S3 record has at most one owner.

---


In [3]:
import pandas as pd
from pathlib import Path
dataset = Path("../../dataset")

## Data loading

In [4]:
gt = pd.read_csv(dataset/"train"/"train_ground_truth.tsv", sep= '\t')
s1 = pd.read_csv(dataset/"train"/"train_source1.tsv", sep= '\t')
s2 = pd.read_csv(dataset/"train"/"train_source2.tsv", sep= '\t')
s3 = pd.read_csv(dataset/"train"/"train_source3.tsv", sep= '\t')

In [5]:
s1.head(10)

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US


In [6]:
s2.head(10)

,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US
5,S2-138046867,Lee and Lawson,"1702 Pine Avenue, CITY OF MENOMONIE, WI",US
6,S2-584977605,SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPOR...,"H.NO 204 C ROAD HOSHIARPUR, PUNJAB, Punjab",India
7,S2-277444929,Shree Infracon Private Ltd,"63/2275/7, ALHIND TOWER, FIRST FLOOR, JAFFERKH...",India
8,S2-721031885,Chavira Platinum Chimera LLC,"282 SAXONY DRIVE, FTT MITCHELL, KY",US
9,S2-508602797,FOUNDATION EXCEL AGENCY PRIVATE LIMITED,"HN 753 E-1, BHARAT NAGAR, 104/1/1 ERANDWANE, M...",India


In [7]:
s3.tail(10)

,entity_id,business_name,business_address,country
5285593,S3-249255340,Conner & LINZ Biosciences INC,"31 Woodfin Ave, Asheville, North Carolina",US
5285594,S3-285629294,Hitech Infrastructure Engineering Private Limited,"WB, Jalpaiguri, No 35 C/o Satvikk Agarwal, Sil...",India
5285595,S3-537793423,અલ એન્ટરપ્રાઇઝિસ પ્રાઇવેટ લિમિટેડ,"D-704 Prince Heaven Near, Elephanta Business H...",India
5285596,S3-268760762,Interstate Corporation,"00301 Park Ave, Renton, Washington",US
5285597,S3-355823625,Solgild Labs D.B.A. GH Certified Dental LLC,"20019 Spring Hill Lane, Orange County, VA",US
5285598,S3-549653960,Vip Solutions,NaN,India
5285599,S3-45212707,Cornerstone Nippon,"Paula Drive, Arkansas, N Little Rock",US
5285600,S3-257390113,Legacy Piedmont,"Portland, Maine, Forest Park",US
5285601,S3-97881155,Pranava Meribl Trust,"235, 13Th Cross Road, Hoyasala Ngr 2D Stage, B...",India
5285602,S3-864344230,The Trusted Defense Products Inc,NaN,US


In [8]:
s3.shape

(5285603, 4)

In [9]:
s3["business_name"].isna().sum()

np.int64(13)

## Structural Analysis

In [25]:
"""
Section A -- structural verification (lean).

Core question: did the files parse the way we think they did?
Written for a notebook -- each function returns a DataFrame you can just
display. Script-mode equivalents are commented out at the bottom.
"""
import csv
import re
from collections import Counter
from pathlib import Path

DATA_DIR = Path("../../dataset")
COLUMNS = ["entity_id", "business_name", "business_address", "country"]
ID_RE = re.compile(r"^S([123])-\d+$")

# Cells pandas would silently turn into NaN on a default read. We read with
# na_filter=False so they survive; this set is only used to count them.
NA_LOOKALIKES = {"", "NA", "N/A", "NaN", "nan", "NULL", "null", "None", "<NA>"}


def source_paths(split):
    return {f"S{n}": DATA_DIR / split / f"{split}_source{n}.tsv" for n in (1, 2, 3)}


def read_source(path):
    """Read without letting pandas invent missingness.

    dtype=str + na_filter=False -> 'NA Enterprises' and a blank cell stay
    distinguishable. QUOTE_NONE -> quotes inside addresses stay literal.
    """
    return pd.read_csv(
        path, sep="\t", dtype=str, na_filter=False, keep_default_na=False,
        quoting=csv.QUOTE_NONE, encoding="utf-8-sig",
    )


def scan_lines(path):
    """Stream the file and count tabs per line. Catches rows pandas shifted
    because a field contained a literal tab."""
    counts = Counter()
    with open(path, encoding="utf-8-sig", errors="replace") as f:
        header = next(f, "")
        for line in f:
            counts[line.rstrip("\n").count("\t")] += 1
    return header.rstrip("\n"), counts


def structure_row(name, path):
    """One row of the top-level summary table."""
    header, tab_counts = scan_lines(path)
    df = read_source(path)
    ids = df["entity_id"]
    prefix_ok = ids.str.extract(ID_RE)[0] == name[1]

    return {
        "source": name,
        "raw_lines": sum(tab_counts.values()),
        "parsed_rows": len(df),
        "header_ok": header.split("\t") == COLUMNS,
        "bad_tab_lines": sum(v for k, v in tab_counts.items() if k != 3),
        "ids_unique": ids.nunique() == len(ids),
        "ids_malformed": int((~ids.str.match(ID_RE)).sum()),
        "ids_wrong_prefix": int((~prefix_ok.fillna(False)).sum()),
        "exact_dup_rows": int(df.duplicated().sum()),
        "dup_name_addr_country": int(df.duplicated(COLUMNS[1:], keep=False).sum()),
        "countries": dict(df["country"].value_counts()),
    }


def structure_summary(split):
    """Top-level table: one row per source file."""
    return pd.DataFrame(
        structure_row(n, p) for n, p in source_paths(split).items() if p.exists()
    )


def blank_profile(df):
    """Split 'missing' into its literal forms, per field.

    This is what separates a truly blank cell from the string 'NaN' sitting in
    the source file from a business actually named 'NA'.
    """
    rows = []
    for col in ("business_name", "business_address"):
        vals = df[col].str.strip()
        counts = vals[vals.isin(NA_LOOKALIKES)].value_counts()
        for literal, n in counts.items():
            rows.append({"field": col, "literal": literal or "<empty>", "count": n})
        rows.append({"field": col, "literal": "TOTAL",
                     "count": int(vals.isin(NA_LOOKALIKES).sum())})
    return pd.DataFrame(rows)


def duplicate_examples(df, n=10):
    """Content duplicates ignoring the ID. For S1 this tests the
    'deduplicated reference source' claim -- verify it, don't assume it."""
    mask = df.duplicated(COLUMNS[1:], keep=False)
    dups = df.loc[mask].sort_values(COLUMNS[1:])
    return dups.head(n), dups.shape


# --------------------------------------------------------------------------
# Notebook usage
# --------------------------------------------------------------------------
# summary = structure_summary("train"); summary
# s1 = read_source(source_paths("train")["S1"])
# blank_profile(s1)
# duplicate_examples(s1)

# --------------------------------------------------------------------------
# Script mode -- uncomment when this moves into the pipeline
# --------------------------------------------------------------------------
# if __name__ == "__main__":
#     for split in ("train", "test"):
#         if not (DATA_DIR / split).exists():
#             continue
#         summary = structure_summary(split)
#         print(f"\n=== {split} ===")
#         print(summary.to_string(index=False))
#         summary.to_csv(f"reports/section_a_{split}.tsv", sep="\t", index=False)
#         for name, path in source_paths(split).items():
#             df = read_source(path)
#             print(f"\n[{name}] blanks")
#             print(blank_profile(df).to_string(index=False))

In [ ]:
# --------------------------------------------------------------------------
# Notebook usage
# --------------------------------------------------------------------------
summary = structure_summary("train"); summary
s1 = read_source(source_paths("train")["S1"])


,entity_id,business_name,business_address,country


In [26]:
dups_1, dups_1_shape = duplicate_examples(s1)
dups_1.head()

,entity_id,business_name,business_address,country


In [27]:
dups_1_shape

(0, 4)

In [13]:
blank_profile(s1)

,field,literal,count
0,business_name,TOTAL,0
1,business_address,TOTAL,0


In [14]:
s2 = read_source(source_paths("train")["S2"])
blank_profile(s2)

,field,literal,count
0,business_name,NA,2
1,business_name,TOTAL,2
2,business_address,<empty>,168967
3,business_address,TOTAL,168967


In [28]:
dups_2, dups_2_shape = duplicate_examples(s2)
dups_2.head()

,entity_id,business_name,business_address,country
1173331,S2-925640291,# 2 CE Wellness,"5615 MONROE ST, HYATTSVILLE CITY, MD",US
2114519,S2-162845527,# 2 CE Wellness,"5615 MONROE ST, HYATTSVILLE CITY, MD",US
876385,S2-463843468,# 2 KY Associated Llc,"1510- BEECHER DRIVE, SAINT LOUIS, MO",US
5014942,S2-515056361,# 2 KY Associated Llc,"1510- BEECHER DRIVE, SAINT LOUIS, MO",US
84453,S2-267996615,"# 6 CO Bsp,","425 587, SMITHS STATION, AL",US


In [29]:
dups_2_shape

(50933, 4)

In [16]:
s3 = read_source(source_paths("train")["S3"])
blank_profile(s3)

,field,literal,count
0,business_name,NA,13
1,business_name,TOTAL,13
2,business_address,<empty>,175916
3,business_address,TOTAL,175916


In [30]:
dups_3, dups_3_shape = duplicate_examples(s3)
dups_3.head()

,entity_id,business_name,business_address,country
6699,S3-683879345,# 3 AH Nasdaq,"11686 35th Street, Yuma, Arizona",US
4152765,S3-216952998,# 3 AH Nasdaq,"11686 35th Street, Yuma, Arizona",US
1927912,S3-441978577,# 7 OF Eqv LLC,,US
5081440,S3-52513033,# 7 OF Eqv LLC,,US
69776,S3-196670208,"# 8 PU Stock,","##3027 Hickory Grove Ct, Fairfax, Virginia",US


In [31]:
dups_3_shape

(37241, 4)

In [32]:
s3['business_name'].str.contains('\xa0').sum()

np.int64(0)

checking group count and size distribution

In [38]:
def dup_groups(df):
    g = df.groupby(COLUMNS[1:]).size()
    g = g[g > 1]
    return g.value_counts().sort_index() #len(g)

s2_val = dup_groups(s2)
s3_val = dup_groups(s3)

In [39]:
s2_val

2    24270
3      769
4       19
5        2
Name: count, dtype: int64

In [40]:
s3_val

2    17919
3      445
4       17
Name: count, dtype: int64

In [41]:
gt = pd.read_csv(DATA_DIR/"train"/"train_ground_truth.tsv", sep="\t",
                 dtype=str, na_filter=False, keep_default_na=False,
                 quoting=csv.QUOTE_NONE)

pairs = gt.assign(mid=gt["matched_entity_ids"].str.split(",")).explode("mid")
pairs = pairs[pairs["mid"].str.strip() != ""]
owner = pairs.groupby("mid")["source1_entity_id"].first()

In [42]:
dup = s2[s2.duplicated(COLUMNS[1:], keep=False)].copy()
dup["grp"] = dup.groupby(COLUMNS[1:]).ngroup()
dup["owner"] = dup["entity_id"].map(owner)

check = dup.groupby("grp")["owner"].agg(
    members="size",
    owners=lambda s: s.nunique(),
    unmatched=lambda s: s.isna().sum(),
)
check.value_counts(["owners", "unmatched"]).head(10)

owners  unmatched
1       0            25060
Name: count, dtype: int64

In [43]:
dup = s3[s3.duplicated(COLUMNS[1:], keep=False)].copy()
dup["grp"] = dup.groupby(COLUMNS[1:]).ngroup()
dup["owner"] = dup["entity_id"].map(owner)

check = dup.groupby("grp")["owner"].agg(
    members="size",
    owners=lambda s: s.nunique(),
    unmatched=lambda s: s.isna().sum(),
)
check.value_counts(["owners", "unmatched"]).head(10)

owners  unmatched
1       0            18381
Name: count, dtype: int64

## Section A — Structural Verification: Findings

**Files are clean.** Headers correct, line counts reconcile, no embedded tabs. `entity_id` unique and well-formed in all three sources. S1 confirmed deduplicated — the problem statement's claim holds.

**Missingness is one phenomenon.** ~169k empty addresses in S2, ~176k in S3 (~3% each). No literal `"NaN"`/`"null"` text anywhere. The earlier "thousands of NaN" count was a default-`read_csv` artefact. Blank names negligible (2 in S2, 13 in S3, all the string `NA`).
→ Read strictly everywhere. Carry `has_address`; don't impute, don't drop. Matcher must fall back to name-only scoring when address is absent. ~3% of S2/S3 can't use address-based blocking keys — a hard recall ceiling on that slice.

**S2/S3 contain injected duplicates; S1 doesn't.** Identical `(name, address, country)`, different IDs: 50,933 rows / 25,060 groups in S2; 37,241 rows / 18,381 groups in S3. Mostly pairs. Ground-truth check returns `owners=1, unmatched=0` for **every** group — all members claimed by the same S1 entity, none unlabelled. Zero singletons among ~88k duplicate rows means these were injected deliberately, not coincidental collisions.
→ Collapse to one representative per content hash before blocking (~44k fewer inferences, no recall cost), then **expand predictions back through the hash→IDs map before writing output**. Omitting siblings loses recall for free.

**Per-source address conventions are systematic.** S2: `5615 MONROE ST, HYATTSVILLE CITY, MD` — ALLCAPS, abbreviated street type, two-letter state, `CITY` suffix. S3: `11686 35th Street, Yuma, Arizona` — Title Case, full street type, full state name.
→ Casefold; expand street types bidirectionally; map state code ↔ name; strip trailing `CITY`. Derive both vocabularies from `value_counts` on trailing comma-segments rather than hardcoding — fair-play clean and generalizes to France in test. Blocking keys must be built on the normalized form.

**Character noise.** Double spaces in names (not NBSP — `str.contains('\xa0')` is 0; the `&nbsp;` in notebook output is a pandas HTML artefact). Leading `#`+digit tokens (`# 2`, `#5`). Trailing commas on names. `##3027` doubled hash. `1510-` trailing hyphen. Bare-digit addresses (`425 587`).
→ Whitespace collapse + punctuation strip first. Quantify the `#`+digit pattern before stripping — check whether matched S1 counterparts retain it. Fold bare-digit addresses into `has_address = False`.

**Open for later:** Is S1 entirely Latin-script? (Section C — decides if transliteration is one-way.) Overall singleton rate? (Section F — sets precision/recall posture under macro-F₀.₅.)

## Downstream dictation
- ~3% of S2/S3 cannot participate in any address-derived blocking key. They need a name-only key path or they are unreachable — a hard recall ceiling on that slice.
- The matcher must branch on has_address: a missing address must trigger name-only scoring, not a 0.0 address-similarity score that kills an otherwise good pair.
- Add a content-hash column over normalized (name, address, country) for S2 and S3.
- Build a hash → [entity_ids] expansion map and persist it alongside the frames.
- Select one representative row per hash group as the canonical record.
- Block and score representatives only — ~44k fewer records through candidate generation and model inference, at zero recall cost.
- Expand predictions back through the map before writing output. A predicted match on a representative means every member ID of its group is a match. 
- Omitting siblings loses recall for free; under macro-F₀.₅ that is unrecovered credit on those entities.
- Twin consistency becomes structural rather than something the model must rediscover pair-by-pair.
- Apply the identical collapse/expand logic to the test set. Verify the duplicate rate holds there before relying on it.
- Casefold before any comparison.
- Bidirectional street-type expansion (ST↔Street, RD↔Road, …).
- State-code ↔ state-name mapping.
- Strip trailing CITY from locality segments.
- Derive both vocabularies from the corpus itself (value_counts on trailing comma-segments), not from a hardcoded US list. Fair-play clean, and it generalizes to France in the test set without modification.
- A raw-string blocking key would place S2 and S3 versions of the same business in different blocks. Keys must be built on the normalized form.
- Convention differences are cross-source and systematic, so per-source normalization must converge on one canonical representation before keys are generated.
- Whitespace collapse + strip as the first normalization pass.
- Strip leading/trailing punctuation from names.
- Quantify the leading #+digit pattern before deciding whether to strip it — if it is a marker it is noise; if it is part of the trade name it is signal. Check whether matched S1 counterparts retain it.
- Count the bare-digit address shape; those records are effectively address-less and should be folded into the has_address = False path.

### Checking degenerate values of name/address

In [44]:
def degenerate(df, col, maxlen=3):
    """Short or content-free values that aren't empty strings."""
    s = df[col].str.strip()
    short = s[(s != "") & (s.str.len() <= maxlen)]
    no_alnum = s[(s != "") & (~s.str.contains(r"[^\W_]", regex=True))]
    return short.value_counts(), no_alnum.value_counts()

In [45]:
for name, df in [("S1", s1), ("S2", s2), ("S3", s3)]:
    for col in ["business_name", "business_address"]:
        short, junk = degenerate(df, col)
        print(f"\n{name} {col}  short={short.sum()} punct_only={junk.sum()}")
        print(short.head(10))


S1 business_name  short=561 punct_only=0
business_name
Boa    20
Zio    11
Mio    10
Fao    10
Bia     9
Baa     9
Ria     8
Saa     8
Boo     8
Daa     8
Name: count, dtype: int64

S1 business_address  short=0 punct_only=0
Series([], Name: count, dtype: int64)

S2 business_name  short=2946 punct_only=0
business_name
Boa    21
Eye    21
Vio    10
Fao    10
Baa    10
Mia    10
Pea    10
Doo     9
Tia     9
Kea     9
Name: count, dtype: int64

S2 business_address  short=0 punct_only=0
Series([], Name: count, dtype: int64)

S3 business_name  short=17006 punct_only=0
business_name
SC     197
SI     180
SS     160
ST     146
SE     123
AC     115
SP     109
Eye    108
Ace    107
All    104
Name: count, dtype: int64

S3 business_address  short=1 punct_only=0
business_address
DL    1
Name: count, dtype: int64


In [52]:
def unusable(df):
    n = df["business_name"].str.strip()
    a = df["business_address"].str.strip()
    weak_name = (n == "") | (n.str.len() <= 2) | (~n.str.contains(r"[^\W_]"))
    return (weak_name & (a == "")).sum()

In [53]:
unusable(s1)

np.int64(0)

In [54]:
unusable(s2)

np.int64(3)

In [55]:
unusable(s3)

np.int64(4)

In [56]:
short_ids = s3.loc[s3["business_name"].str.strip().str.len() <= 2, "entity_id"]
short_ids.isin(owner.index).mean()

np.float64(0.9990296495956873)

In [57]:
short_ids = s2.loc[s2["business_name"].str.strip().str.len() <= 2, "entity_id"]
short_ids.isin(owner.index).mean()

np.float64(0.9801136363636364)

In [58]:
chk = s3.loc[s3["business_name"].str.strip().str.len() <= 2, ["entity_id", "business_name"]]
chk["s1"] = chk["entity_id"].map(owner)
chk = chk.merge(s1[["entity_id", "business_name"]], left_on="s1",
                right_on="entity_id", suffixes=("_s3", "_s1"))
chk[["business_name_s3", "business_name_s1"]].head(20)

,business_name_s3,business_name_s1
0,GM,Gujju Media LLP
1,AC,Aditya Consulting Private Limited
2,LC,La Club Limited
3,IE,Indo Estate Private Limited
4,SS,Supreme Solutions Private Limited
5,SO,Sudduth and Olea Corp
6,Zx,Zaix
7,BT,Bharat Technology Limited
8,RM,Rapid Mobility Corp
9,ME,Maniratnam Ecommerce Private Limited


In [59]:
chk_ids = set(chk["entity_id_s3"])
(s3[s3["entity_id"].isin(chk_ids)]["business_address"].str.strip() == "").mean()

np.float64(0.0004316857327865314)

In [61]:
chk = s2.loc[s2["business_name"].str.strip().str.len() <= 2, ["entity_id", "business_name"]]
chk["s1"] = chk["entity_id"].map(owner)
chk = chk.merge(s1[["entity_id", "business_name"]], left_on="s1",
                right_on="entity_id", suffixes=("_s2", "_s1"))
chk[["business_name_s2", "business_name_s1"]].head(20)

,business_name_s2,business_name_s1
0,MT,Mccloud's Trading
1,MÍ,Morales Innovative LLC
2,CS,Career Services Limited
3,SI,Si Group
4,Qm,Quum
5,IC,Innovision Consultants
6,#K,K & Y State
7,SG,Silver Gamco Corporation
8,WC,West Cartesian Inc
9,MP,Mccabe Praetorian Inc


In [62]:
SUFFIX = {"llp","llc","ltd","limited","pvt","private","inc","corp","corporation",
          "co","company","and","&","the","of","plc","pc","group"}

def initialism(name):
    toks = re.findall(r"[^\W\d_]+", name.lower())      # letters only, drops numbers
    toks = [t for t in toks if t not in SUFFIX]
    return "".join(t[0] for t in toks)

def locality(addr):
    parts = [p.strip().lower() for p in addr.split(",") if p.strip()]
    if len(parts) < 2:
        return ""
    return re.sub(r"\s+city$", "", parts[-2])          # second-last ≈ city

In [65]:
k = pd.DataFrame({
    "init": s1["business_name"].map(initialism),
    "loc":  s1["business_address"].map(locality),
})
k = k[(k["init"].str.len() > 0) & (k["loc"] != "")]

sizes = k.groupby(["init", "loc"]).size()
print(sizes.value_counts().sort_index().head(10))
print("mean:", sizes.mean(), "p99:", sizes.quantile(0.99))

1     1250853
2       94849
3       30616
4       14600
5        8717
6        5485
7        3865
8        2764
9        2056
10       1694
Name: count, dtype: int64
mean: 1.5446359952123974 p99: 10.0


In [66]:
two = sizes[sizes.index.get_level_values("init").str.len() == 2]
print(two.value_counts().sort_index().head(10))
print("p50:", two.median(), "p99:", two.quantile(0.99), "max:", two.max())

1     385413
2      53473
3      20173
4      10443
5       6466
6       4217
7       3021
8       2238
9       1670
10      1397
Name: count, dtype: int64
p50: 1.0 p99: 23.0 max: 1167


In [69]:
def house_no(addr):
    m = re.match(r"\s*#?\s*(\d+)", addr)
    return m.group(1) if m else ""

k["house"] = s1.loc[k.index, "business_address"].map(house_no)
sizes3 = k[k["house"] != ""].groupby(["init", "loc", "house"]).size()
two3 = sizes3[sizes3.index.get_level_values("init").str.len() == 2]
print("p99:", two3.quantile(0.99), "max:", two3.max())
print((k["house"] != "").mean())

p99: 2.0 max: 29
0.6222327653631539


In [70]:
gap = s1.loc[k.index[k["house"] == ""], "country"]
print(gap.value_counts(normalize=True))
print(s1["country"].value_counts(normalize=True))   # baseline for comparison

country
India    0.777314
US       0.222686
Name: proportion, dtype: float64
country
US       0.599792
India    0.400208
Name: proportion, dtype: float64


In [71]:
s1.loc[k.index[k["house"] == ""], "business_address"].head(20).tolist()

['OH, Columbus, 5559 Orville Avenue',
 'H.No.16-11-23/37/A, 2Nd Floor, Flat No.207, Sagar Hotel Building, Opp.Rta Office, Mo, Osarambagh, Hyderabad, Telangana',
 'Unit BUILDING 3030, MD, 2701 Eastern Boulevard, Middle River',
 'Charlotte, NC, 833 Reliance Street',
 'House No-777 Raghubir Bhawanmadan Pur Khadar Sarita Vihar, Delhi, South Delhi, Delhi',
 'Secunderabad, Hyderabad, Plot No.53, Flat No.402, Sr Estate-I Rainbow Colony, Ammuguda, Sainikpuri, Telangana, # 53/1',
 'Sapulpa, OK, 2204 Park Street',
 'E-7, Second Floor, New Delhi, South Delhi, Delhi',
 'Building No. 37/4, First Floor Opp Bharata Mata College, Vazhakkala, Thrikkakara, Ernakulam, Kerala',
 'C/O Dwarkadhis Enterprise, C.M. Palace, Ist Floor, Shop 3, Vill: Gondal(Mog), Gondal, Rajkot, Gondal, Rajkot, Gujarat',
 'Unit UNIT 367, 1400 Great Wolf Drive, WI, Village Of Lake Delton',
 'House No.26-352, Kadavakollu Bhavanam, Bhuttaipeta, Near E.N.T Hospital, Machilipatanam, Krishna, Andhra Pradesh',
 'D No. 20-6-3/12, Dantul

In [72]:
HOUSE = re.compile(
    r"^[\s#]*(?:h\.?\s*no\.?|kh\.?\s*no\.?|d\.?\s*no\.?|plot\s*no\.?|plot|flat|shop|survey\s*no\.?)?"
    r"[\s.:\-]*([0-9]+(?:[-/][0-9A-Za-z]+)*)",
    re.I,
)

def house_no2(addr):
    m = HOUSE.match(addr)
    return m.group(1).lower() if m else ""

k["house2"] = s1.loc[k.index, "business_address"].map(house_no2)
print("coverage:", (k["house2"] != "").mean())

coverage: 0.6539224562557833


In [73]:
STOP = {"near","opp","opposite","behind","beside","no","floor","road","street","st","rd"}

def street_tok(addr):
    parts = [p.strip() for p in addr.split(",") if p.strip()]
    if not parts:
        return ""
    toks = [t for t in re.findall(r"[^\W\d_]+", parts[0].lower()) if t not in STOP]
    return max(toks, key=len) if toks else ""      # longest token ≈ most distinctive

rest = k[k["house2"] == ""].copy()
rest["stok"] = s1.loc[rest.index, "business_address"].map(street_tok)
sz = rest[rest["stok"] != ""].groupby(["init", "loc", "stok"]).size()
sz2 = sz[sz.index.get_level_values("init").str.len() == 2]
print("p99:", sz2.quantile(0.99), "max:", sz2.max(), "coverage:", (rest["stok"] != "").mean())

p99: 4.0 max: 55 coverage: 0.960061592770182


## Section B — Degenerate Values & Short Names: Findings

### Degenerate values are a non-issue
- `punct_only = 0` in all three sources; addresses have essentially no short values (one `DL` in S3).
- Truly unmatchable records (degenerate name **and** empty address): 0 / 3 / 4 across S1 / S2 / S3. Nothing needs excluding.
- 3-letter names (`Boa`, `Zio`, `Ace`, `Eye`) are legitimate and appear in S1 too → degeneracy threshold is **≤ 2 chars, not ≤ 3**.

### Short names are initialisms, and they are matchable
- ≤2-char names: 704 in S2, 9,275 in S3 — with **98% / 99.9% having a true S1 match**. Excluding them would have discarded ~10k matchable entities.
- Patterns: token initials skipping suffixes/connectors/numerics (`AC` ← Aditya Consulting Private Limited, `SO` ← Sudduth and Olea, `WD` ← 864 Washington Drive Company); first+last letter of single-token names, Title-cased (`Zx` ← Zaix, `Px` ← Peex); junk prefixes (`#K`, `@c`); diacritics (`MÍ` ← Morales Innovative). Small unrecoverable residue (`Ae` ← Atlas).

**→ Normalization**
- NFKD + accent strip (confirmed needed; more so for France in test).
- Strip leading junk punctuation from names.
- Derive two keys per S1 record: token-initialism (suffix/connector/numeric-aware) and first+last-letter form, both casefolded. Suffix vocabulary to be derived empirically in Section D, not hardcoded.

**→ Blocking**
- A 2-char name is unreachable by any string-similarity metric → this slice needs the key **inverted: address blocks, name confirms**. Viable since only 0.04% of short-name S3 records lack an address.
- `s1_initialism == s2_name` becomes a near-decisive model feature for this slice.

### Blocking key granularity (S1, 2-char initialisms)
| Key | p99 | max | coverage |
|---|---|---|---|
| `(init, locality)` | 23 | 1,167 | — |
| `(init, locality, house_no)` | 2 | 29 | 65% |
| `(init, locality, street_token)` on the remainder | 4 | 55 | 96% of rest |

- `(init, locality)` alone is unusable — 10+ candidates against a 2-letter name destroys precision, penalised twice under F₀.₅.
- Two-tier key gives **~98.6% combined coverage** at p99 ≤ 4. Section B's open question resolved.

### Critical finding: address component order is not fixed
- The 38% "no house number" gap was **not missing numbers — it was permuted addresses**. The regex only looked at position 0.
- `OH, Columbus, 5559 Orville Avenue` (state, city, street) vs `Charlotte, NC, 833 Reliance Street` (city, state, street) vs `West Bengal, Howrah, 229, Kolkata, …`.
- India is 78% of the gap vs a 40% corpus baseline (~2.3× over-represented) because Indian addresses genuinely lead with door/building identifiers — but the US 22% is pure reordering.

**→ Normalization (supersedes earlier positional assumptions)**
- **No positional parsing.** No component sits at a fixed index — not street number, city, state, or postcode.
- Extract component **sets**, not ordered fields: all numeric tokens via `re.findall(r"\d+", addr)`, all alpha tokens, recognised state/locality tokens. Compare as sets.
- Locality extractor using `parts[-2]` is unsafe — city appears at index 1, 3, or elsewhere.
- Section E revision: build locality vocabulary from `value_counts` over **all** comma-segments, then match any segment against it.

**→ Blocking**
- Numeric key = intersection of numeric-token sets, position-independent. Recovers `229` mid-string and trailing `# 53/1`.
- House-number regex widening only gained 62.2% → 65.4%; the order problem, not the regex, was the real limit.

### Additional noise for the rule list
- Doubled unit markers: `Unit BUILDING 3030`, `Unit UNIT 367`
- Mangled ordinals from naive title-casing: `2Nd`, `Ist`, `3Rd` → normalize to one form
- Colon separators and parentheticals: `Vill: Gondal(Mog)`
- Care-of prefixes: `C/O Dwarkadhis Enterprise`
- Landmark markers with no space after punctuation: `Opp.Rta Office`, `Near E.N.T Hospital`
- Bare-digit-pair addresses recur: `AL, Hodges, 541 117`, `425 587`

## Language differences

In [75]:
import regex as re2

def script_of(s):
    if re2.search(r"\p{Devanagari}", s): return "devanagari"
    if re2.search(r"\p{Gujarati}", s):   return "gujarati"
    if re2.search(r"[^\p{Latin}\p{Common}]", s): return "other"
    return "latin"

for n, df in [("S1", s1), ("S2", s2), ("S3", s3)]:
    print(n, df["business_name"].map(script_of).value_counts(normalize=True).round(4).to_dict())

S1 {'latin': 1.0}
S2 {'latin': 0.9058, 'devanagari': 0.0535, 'other': 0.0346, 'gujarati': 0.0061}
S3 {'latin': 0.9473, 'devanagari': 0.0299, 'other': 0.0194, 'gujarati': 0.0034}


## suffix vocab

In [76]:
toks = s1["business_name"].str.lower().str.findall(r"[^\W\d_]+")
print(toks.str[-1].value_counts().head(30))   # trailing
print(toks.str[0].value_counts().head(20))    # leading

business_name
limited        521915
llc            355736
inc            238306
ltd            148595
c               45559
llp             39725
corp            33761
group           32678
pc              25827
co              25191
center          22269
associates      22165
pllc            20531
partners        19351
clinic          18107
care            16845
lp              16477
corporation     13540
company         12316
trust            9924
holdings         8309
society          7024
school           5913
foundation       5313
church           5114
d                5017
medicine         4470
services         4317
academy          4167
institute        4090
Name: count, dtype: int64
business_name
pediatric    15058
new          10180
blue          9145
vision        8624
global        8400
golden        8215
silver        7807
bright        7596
green         7457
first         7407
shree         7046
united        6772
family        6688
great         6617
premier       6531
s

## locality vocabulary

In [77]:
segs = pd.Series([p.strip().lower() for a in s1["business_address"] for p in a.split(",") if p.strip()])
print(segs.value_counts().head(40))

maharashtra      191971
delhi            155370
tx               133202
ny               102380
nc                94673
oh                85246
mumbai            81293
new delhi         76725
il                75665
uttar pradesh     75174
karnataka         70306
tamil nadu        63482
bangalore         60749
tn                59661
va                59608
gujarat           57653
west bengal       57007
ma                56997
az                56535
telangana         56189
kolkata           54271
in                49472
hyderabad         47645
pune              44892
wa                42211
md                40659
mumbai city       37843
ca                37402
haryana           36714
kerala            35661
al                32688
chennai           32535
rajasthan         32358
wi                31434
mn                31138
or                29859
howrah            29632
ky                29478
thane             27876
ar                27504
Name: count, dtype: int64


In [78]:
pat = r"\b(near|opp|opposite|behind|beside|adjacent|back side|in front of)\b|opp\."
print(s1["business_address"].str.lower().str.contains(pat, regex=True).mean())

C:\Users\Khalid Mohammad\AppData\Local\Temp\ipykernel_43632\1283019205.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print(s1["business_address"].str.lower().str.contains(pat, regex=True).mean())


0.045727315446064724


In [82]:
"""
Section F -- Ground-truth-conditioned EDA.

Everything before this described the data. This describes the PROBLEM:
how many matches per entity, how separable true pairs are from random pairs,
and what recall ceiling each blocking key can reach.

Notebook-first. Assumes s1, s2, s3 are already loaded via read_source(),
and that `owner` (matched_id -> source1_entity_id) exists from Section A.
"""

import re
from collections import Counter

import numpy as np
import pandas as pd
from rapidfuzz import fuzz

RNG = np.random.default_rng(0)
SAMPLE = 20000          # pairs sampled for the similarity overlays


# ---------------------------------------------------------------- setup ----

def load_gt(path="../../dataset/train/train_ground_truth.tsv"):
    import csv
    return pd.read_csv(path, sep="\t", dtype=str, na_filter=False,
                       keep_default_na=False, quoting=csv.QUOTE_NONE,
                       encoding="utf-8-sig")


def explode_gt(gt):
    """One row per (s1_id, matched_id) pair. Drops empties."""
    p = gt.assign(mid=gt["matched_entity_ids"].str.split(",")).explode("mid")
    p["mid"] = p["mid"].str.strip()
    return p[p["mid"] != ""][["source1_entity_id", "mid"]].reset_index(drop=True)


def build_lookup(s1, s2, s3):
    """entity_id -> (name, address, country) for every record, one dict-frame."""
    allrec = pd.concat([s1, s2, s3], ignore_index=True)
    return allrec.set_index("entity_id")[["business_name", "business_address", "country"]]


# ------------------------------------------------- F1: match structure ----

def match_structure(gt, s1):
    """Singleton rate and match-count distribution -- the numbers that set
    the whole precision/recall posture under macro-F0.5."""
    lists = gt["matched_entity_ids"].apply(
        lambda s: [t for t in (p.strip() for p in s.split(",")) if t])
    n = lists.str.len()

    dist = n.value_counts().sort_index()
    print("match-count distribution:")
    print(dist.head(12).to_string())
    print(f"\nentities           : {len(gt)}")
    print(f"singleton rate     : {(n == 0).mean():.4f}")
    print(f"mean matches       : {n.mean():.3f}")
    print(f"p95 / max          : {n.quantile(0.95):.0f} / {n.max()}")
    print(f"\nceiling if predict ALL empty : {(n == 0).mean():.4f}")
    return n


def source_split(pairs):
    """Do matches come from S2, S3, or both?"""
    pairs = pairs.copy()
    pairs["src"] = pairs["mid"].str[:2]
    print("\nmatched ids by source:")
    print(pairs["src"].value_counts().to_string())

    per = pairs.groupby("source1_entity_id")["src"].agg(set)
    combo = per.map(lambda s: "+".join(sorted(s)))
    print("\nper-entity source combination:")
    print(combo.value_counts().to_string())
    return combo


# ------------------------------------------- F2: similarity separability --

def norm(s):
    s = s.lower()
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def pair_features(a_ids, b_ids, lookup):
    """Vectorised-ish feature block for a list of id pairs."""
    A = lookup.reindex(a_ids)
    B = lookup.reindex(b_ids)
    out = pd.DataFrame(index=range(len(a_ids)))

    an = [norm(x) for x in A["business_name"].fillna("")]
    bn = [norm(x) for x in B["business_name"].fillna("")]
    aa = [norm(x) for x in A["business_address"].fillna("")]
    ba = [norm(x) for x in B["business_address"].fillna("")]

    out["name_ratio"] = [fuzz.ratio(x, y) for x, y in zip(an, bn)]
    out["name_token_set"] = [fuzz.token_set_ratio(x, y) for x, y in zip(an, bn)]
    out["addr_ratio"] = [fuzz.token_set_ratio(x, y) for x, y in zip(aa, ba)]
    out["addr_missing"] = [(not x) or (not y) for x, y in zip(aa, ba)]
    out["country_match"] = (A["country"].values == B["country"].values)
    out["b_short_name"] = [len(y) <= 2 for y in bn]

    # numeric-token overlap, position independent
    out["num_overlap"] = [
        len(set(re.findall(r"\d+", x)) & set(re.findall(r"\d+", y))) > 0
        for x, y in zip(aa, ba)
    ]
    return out


def sample_negatives(pairs, s1, s23_ids, k):
    """Random s1 x (s2|s3) pairs, excluding known true pairs."""
    true = set(zip(pairs["source1_entity_id"], pairs["mid"]))
    a = RNG.choice(s1["entity_id"].values, size=int(k * 1.1))
    b = RNG.choice(s23_ids, size=int(k * 1.1))
    keep = [(x, y) for x, y in zip(a, b) if (x, y) not in true][:k]
    return [x for x, _ in keep], [y for _, y in keep]


def separability(pairs, s1, lookup, n=SAMPLE):
    """Overlay true-pair vs random-pair feature distributions.
    The overlap region is what the model has to resolve."""
    pos = pairs.sample(min(n, len(pairs)), random_state=0)
    pf = pair_features(pos["source1_entity_id"].tolist(), pos["mid"].tolist(), lookup)

    s23_ids = lookup.index[lookup.index.str[:2].isin(["S2", "S3"])].values
    na, nb = sample_negatives(pairs, s1, s23_ids, n)
    nf = pair_features(na, nb, lookup)

    print("\n--- TRUE pairs ---")
    print(pf.describe().loc[["mean", "25%", "50%", "75%"]].round(2).to_string())
    print("\n--- RANDOM pairs ---")
    print(nf.describe().loc[["mean", "25%", "50%", "75%"]].round(2).to_string())

    for col in ["name_ratio", "name_token_set", "addr_ratio"]:
        print(f"\n{col}: true p5={pf[col].quantile(0.05):.0f} "
              f"p50={pf[col].median():.0f} | random p95={nf[col].quantile(0.95):.0f}")
    return pf, nf


def hard_positives(pf, pairs, thresh=60):
    """True pairs that string similarity alone would miss.
    These define what the model must recover beyond fuzzy matching."""
    hard = pf[pf["name_token_set"] < thresh]
    print(f"\ntrue pairs with name_token_set < {thresh}: "
          f"{len(hard)} / {len(pf)} = {len(hard)/len(pf):.3f}")
    print(f"  of those, short b-name : {hard['b_short_name'].mean():.3f}")
    print(f"  of those, addr missing : {hard['addr_missing'].mean():.3f}")
    print(f"  of those, num_overlap  : {hard['num_overlap'].mean():.3f}")
    return hard


# ------------------------------------------- F3: blocking recall ceiling --

STOP = {"limited", "llc", "inc", "ltd", "llp", "corp", "pc", "pllc", "lp",
        "co", "corporation", "company", "and", "the", "of"}


def name_tokens(s):
    return {t for t in norm(s).split() if t not in STOP and len(t) > 1}


def trigrams(s):
    s = norm(s).replace(" ", "")
    return {s[i:i+3] for i in range(len(s) - 2)} if len(s) >= 3 else set()


def num_tokens(s):
    return set(re.findall(r"\d+", s))


def blocking_recall(pairs, lookup, n=SAMPLE):
    """For each candidate key, what fraction of TRUE pairs share a key value?
    This is the recall ceiling -- no model can exceed it."""
    p = pairs.sample(min(n, len(pairs)), random_state=0)
    A = lookup.reindex(p["source1_entity_id"])
    B = lookup.reindex(p["mid"])

    keys = {
        "name_token":  (A["business_name"].map(name_tokens).values,
                        B["business_name"].map(name_tokens).values),
        "name_trigram": (A["business_name"].map(trigrams).values,
                         B["business_name"].map(trigrams).values),
        "addr_numeric": (A["business_address"].map(num_tokens).values,
                         B["business_address"].map(num_tokens).values),
        "addr_token":  (A["business_address"].map(name_tokens).values,
                        B["business_address"].map(name_tokens).values),
    }

    rows = []
    hit = {}
    for k, (av, bv) in keys.items():
        h = np.array([bool(x & y) for x, y in zip(av, bv)])
        hit[k] = h
        rows.append({"key": k, "recall": h.mean()})

    # unions -- what a multi-key blocker would reach
    rows.append({"key": "name_token OR addr_numeric",
                 "recall": (hit["name_token"] | hit["addr_numeric"]).mean()})
    rows.append({"key": "name_trigram OR addr_numeric",
                 "recall": (hit["name_trigram"] | hit["addr_numeric"]).mean()})
    rows.append({"key": "ANY of four",
                 "recall": np.logical_or.reduce(list(hit.values())).mean()})

    out = pd.DataFrame(rows)
    print("\nblocking recall ceiling:")
    print(out.round(4).to_string(index=False))

    missed = ~np.logical_or.reduce(list(hit.values()))
    print(f"\nunreachable by any key: {missed.mean():.4f}")
    return out, p[missed]


# ---------------------------------------------------------- F4: extras ----

def cross_country(pairs, lookup):
    """Do true matches ever cross the country label?"""
    A = lookup.reindex(pairs["source1_entity_id"])["country"].values
    B = lookup.reindex(pairs["mid"])["country"].values
    same = (A == B)
    print(f"\ncross-country true pairs: {(~same).sum()} / {len(same)} = {(~same).mean():.5f}")
    if (~same).sum():
        print(pd.Series([f"{a}->{b}" for a, b in zip(A[~same], B[~same])])
              .value_counts().head().to_string())
    return same


def reuse_check(pairs):
    """Is any S2/S3 id claimed by more than one S1 entity?
    If zero, matching is a clean partition and you can enforce 1-to-1."""
    c = pairs["mid"].value_counts()
    print(f"\nmatched ids claimed by >1 S1 entity: {(c > 1).sum()}")
    return c[c > 1]


# ------------------------------------------------------------ notebook ----
# gt     = load_gt()
# pairs  = explode_gt(gt)
# lookup = build_lookup(s1, s2, s3)
#
# n      = match_structure(gt, s1)
# combo  = source_split(pairs)
# pf, nf = separability(pairs, s1, lookup)
# hard   = hard_positives(pf, pairs)
# rec, missed = blocking_recall(pairs, lookup)
# cross_country(pairs, lookup)
# reuse_check(pairs)

In [83]:
gt     = load_gt()
pairs  = explode_gt(gt)
lookup = build_lookup(s1, s2, s3)
n      = match_structure(gt, s1)
combo  = source_split(pairs)
pf, nf = separability(pairs, s1, lookup)
hard   = hard_positives(pf, pairs)
rec, missed = blocking_recall(pairs, lookup)
cross_country(pairs, lookup)
reuse_check(pairs)

match-count distribution:
matched_entity_ids
0     123247
1     119157
2     375212
3     530841
4     484115
5     321957
6     164868
7      63968
8      18680
9       4205
10       534
11        37

entities           : 2206821
singleton rate     : 0.0558
mean matches       : 3.461
p95 / max          : 6 / 11

ceiling if predict ALL empty : 0.0558

matched ids by source:
src
S3    3944746
S2    3693619

per-entity source combination:
src
S2+S3    1776047
S3        164498
S2        143029

--- TRUE pairs ---
      name_ratio  name_token_set  addr_ratio
mean       79.37           86.27       87.48
25%        72.73           86.96       85.71
50%        87.50          100.00       93.62
75%        96.30          100.00       98.57

--- RANDOM pairs ---
      name_ratio  name_token_set  addr_ratio
mean       31.46           31.89       33.31
25%        26.67           26.47       29.85
50%        31.88           32.00       33.85
75%        37.04           37.29       37.93

name_ratio:

Series([], Name: count, dtype: int64)